## 1. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import anndata as ann

import spatialdm as sdm


import matplotlib.pyplot as plt


## 2. Patch extract_lr to use locally-downloaded CellChatDB files\n(avoids the figshare download failures)

In [ ]:
import spatialdm.main as sdm_main

_original_read_csv = pd.read_csv

def _patched_read_csv(path_or_url, *args, **kwargs):
    local_map = {
        'https://figshare.com/ndownloader/files/36638943': 'interaction_input_CellChatDB.csv',
        'https://figshare.com/ndownloader/files/36638940': 'complex_input_CellChatDB.csv',
    }
    path_or_url = local_map.get(path_or_url, path_or_url)
    return _original_read_csv(path_or_url, *args, **kwargs)

sdm_main.pd.read_csv = _patched_read_csv  # only patches pandas as seen inside main.py


## 3. Load samples\nAll 8 samples, in one consistent order used everywhere below.

In [ ]:
sample_names = ["A1", "A2", "A3", "A4", "A6", "A7", "A8", "A9"]

A1 = ann.read_h5ad("A1.h5ad")
A2 = ann.read_h5ad("A2.h5ad")
A3 = ann.read_h5ad("A3.h5ad")
A4 = ann.read_h5ad("A4.h5ad")
A6 = ann.read_h5ad("A6.h5ad")
A7 = ann.read_h5ad("A7.h5ad")
A8 = ann.read_h5ad("A8.h5ad")
A9 = ann.read_h5ad("A9.h5ad")


## 4. Run SpatialDM's native per-sample pipeline\nweight_matrix -> extract_lr -> global test -> sig_pairs -> local test -> sig_spots, run **separately per sample** (this is the slow-ish ~8 min step, but it's SpatialDM's own lightweight analytical z-score mode, not the heavy part).

In [ ]:
samples_list = [A1, A2, A3, A4, A6, A7, A8, A9]

for adata in samples_list:
    adata.obsm['spatial'] = adata.obsm['spatial'].values
    sdm.weight_matrix(adata, l=75, cutoff=0.2, single_cell=False)          # weight matrix by rbf kernel
    sdm.extract_lr(adata, 'human', min_cell=3)                            # find overlapping LRs from CellChatDB
    sdm.spatialdm_global(adata, 1000, specified_ind=None, method='z-score', nproc=1)   # global Moran selection
    sdm.sig_pairs(adata, method='z-score', fdr=True, threshold=0.1)       # select significant pairs
    sdm.spatialdm_local(adata, n_perm=1000, method='z-score', specified_ind=None, nproc=1)  # local spot selection
    sdm.sig_spots(adata, method='z-score', fdr=False, threshold=0.1)
    print(' done!')


In [ ]:
stat_col = 'z'   # confirm with A1.uns['global_res'].head() — adjust if the column name differs
samples = {
    'A1': A1, 'A2': A2, 'A3': A3, 'A4': A4, 'A8': A8, 'A9': A9,
}
spatialdm_tab = pd.DataFrame({
    name: ad.uns['global_res'][stat_col]
    for name, ad in samples.items()   # your 6 SpatialDM-processed AnnData objects
})
spatialdm_tab.to_csv('spatialdm_results.csv')

In [ ]:
# check for pairs that are probably the same biology but written differently
sd_only = sd_idx - cm_idx - st_idx
cm_only = cm_idx - sd_idx - st_idx

# look for near-matches: same genes, different order/separator
import re
def normalize(name):
    genes = re.split('[_\-\+]', name)
    return '_'.join(sorted(genes))

sd_norm = {normalize(x): x for x in sd_idx}
cm_norm = {normalize(x): x for x in cm_idx}
st_norm = {normalize(x): x for x in st_idx}

normalized_overlap = set(sd_norm) & set(cm_norm) & set(st_norm)
print(len(normalized_overlap), "pairs match after normalizing gene order/separators")

In [ ]:
import numpy as np
import pandas as pd

def fix_object_cols(df):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].where(df[col].notna(), '').astype(str)
    return df

def sanitize(val):
    if isinstance(val, pd.DataFrame):
        return fix_object_cols(val)
    elif isinstance(val, pd.Series):
        return val.to_numpy()
    elif isinstance(val, dict):
        return {k: sanitize(v) for k, v in val.items()}
    elif isinstance(val, np.ndarray) and val.dtype == object:
        return np.array(['' if x is None else x for x in val])
    else:
        return val

def fix_uns(ad):
    for key in list(ad.uns.keys()):
        ad.uns[key] = sanitize(ad.uns[key])
    return ad

In [ ]:
for name, ad in samples.items():
    ad = fix_uns(ad)   # the sanitize-based version from a few messages back
    ad.write_h5ad(f'{name}_spatialdm.h5ad')
    print(f'{name} saved')

## 5. (Optional) Diagnostic: inspect what SpatialDM actually stored\nUseful if you ever need to re-check key names/shapes after a SpatialDM version change.

In [ ]:
print(A1.uns.keys())
print(type(A1.uns['local_z_p']), A1.uns['local_z_p'].shape)
print(type(A1.uns['local_z']), A1.uns['local_z'].shape)


In [ ]:
print(A1.uns['geneInter'].dtypes)
print(A1.uns['geneInter'].isna().sum())

T_TEST

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

samples = {'A1': A1, 'A2': A2, 'A3': A3, 'A4': A4, 'A8': A8, 'A9': A9}
group = {'A1': 'adult', 'A2': 'adult', 'A3': 'fetal', 'A4': 'fetal', 'A8': 'fetal', 'A9': 'fetal'}

stat_col = 'z'   # check with A1.uns['global_res'].head() and adjust
tab = pd.DataFrame({n: ad.uns['global_res'][stat_col] for n, ad in samples.items()})

g1 = [s for s in tab.columns if group[s] == 'adult']
g2 = [s for s in tab.columns if group[s] == 'fetal']

# with only 2 adult samples, require both to have a value
ok = (tab[g1].notna().sum(axis=1) >= 2) & (tab[g2].notna().sum(axis=1) >= 2)
sub = tab[ok]

t, p = stats.ttest_ind(sub[g1], sub[g2], axis=1, equal_var=False)
res = pd.DataFrame({
    'mean_adult': sub[g1].mean(axis=1),
    'mean_fetal': sub[g2].mean(axis=1),
    'diff': sub[g1].mean(axis=1) - sub[g2].mean(axis=1),
    't': t, 'p': p}, index=sub.index)

valid = res['p'].notna()
res.loc[valid, 'fdr'] = multipletests(res.loc[valid, 'p'], method='fdr_bh')[1]
res.sort_values('p').head(20)

T_MODERATED


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats, special, optimize
from statsmodels.stats.multitest import multipletests


def trigamma_inverse(x):
    """Solve trigamma(y) = x for y."""
    if x <= special.polygamma(1, 1e7):
        return np.inf
    return optimize.brentq(lambda y: special.polygamma(1, y) - x, 1e-6, 1e7)


def fit_prior(s2, d):
    """Estimate prior variance s0^2 and prior df d0 from per-pair variances.
    Model: s2_i ~ s0^2 * F(d_i, d0)  (method of moments on log scale)."""
    ok = np.isfinite(s2) & (d > 0)
    s2, d = s2[ok], d[ok]
    s2 = np.maximum(s2, 1e-5 * np.median(s2))        # guard against exact zeros
    e = np.log(s2) - special.digamma(d / 2) + np.log(d / 2)
    emean = e.mean()
    evar = e.var(ddof=1) - np.mean(special.polygamma(1, d / 2))
    if evar > 0:
        d0 = 2 * trigamma_inverse(evar)
    else:
        d0 = np.inf
    if np.isfinite(d0):
        s0sq = np.exp(emean + special.digamma(d0 / 2) - np.log(d0 / 2))
    else:
        s0sq = np.exp(emean)
    return d0, s0sq


def moderated_ttest(tab, g1, g2):
    A = tab[g1].to_numpy(float)
    B = tab[g2].to_numpy(float)
    n1 = np.sum(~np.isnan(A), axis=1)
    n2 = np.sum(~np.isnan(B), axis=1)
    d = n1 + n2 - 2

    # need at least 1 value per group and at least 1 residual df
    keep = (n1 >= 1) & (n2 >= 1) & (d >= 1)
    A, B, n1, n2, d = A[keep], B[keep], n1[keep], n2[keep], d[keep]
    idx = tab.index[keep]

    m1, m2 = np.nanmean(A, axis=1), np.nanmean(B, axis=1)
    sse = (np.nansum((A - m1[:, None]) ** 2, axis=1)
           + np.nansum((B - m2[:, None]) ** 2, axis=1))
    s2 = sse / d                                  # pooled variance per pair
    v = 1 / n1 + 1 / n2                           # variance factor of the mean difference
    diff = m1 - m2

    # empirical Bayes: estimate prior from all pairs, then shrink each variance
    d0, s0sq = fit_prior(s2, d)
    print(f"prior df d0 = {d0:.2f}, prior variance s0^2 = {s0sq:.3f}")

    if np.isfinite(d0):
        s2_mod = (d0 * s0sq + d * s2) / (d0 + d)
        df_total = d0 + d
    else:
        s2_mod = np.full_like(s2, s0sq)
        df_total = np.full_like(s2, np.inf)

    t_mod = diff / np.sqrt(s2_mod * v)
    p_mod = 2 * stats.t.sf(np.abs(t_mod), df_total)

    # ordinary (unmoderated) t for comparison
    t_ord = diff / np.sqrt(s2 * v)
    p_ord = 2 * stats.t.sf(np.abs(t_ord), d)

    res = pd.DataFrame({
        'mean_g1': m1, 'mean_g2': m2, 'diff': diff,
        't_ordinary': t_ord, 'p_ordinary': p_ord,
        't_moderated': t_mod, 'p_moderated': p_mod,
    }, index=idx)
    res['fdr'] = multipletests(res['p_moderated'], method='fdr_bh')[1]
    return res


res_mod = moderated_ttest(tab, g1, g2)     # g1 = adult, g2 = fetal
res_mod.sort_values('p_moderated').head(20)

In [ ]:
res_mod.loc[['EFNB2_EPHB3', 'MIF_CD74_CD44', 'PECAM1_PECAM1'],
            ['diff', 't_ordinary', 't_moderated', 'fdr']]

SAM 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# ---------- 1. Data: complete cases only ----------
cols = g1 + g2
tab_c = tab[cols].dropna()                      # SAM needs a value in every sample
X = tab_c.to_numpy(float)
n_pairs, n_samp = X.shape
obs_mask = np.array([True] * len(g1) + [False] * len(g2))
print(f"{n_pairs} LR pairs kept, {len(tab) - n_pairs} dropped (missing in >=1 sample)")


# ---------- 2. Mean difference and standard error for a given labelling ----------
def diff_and_se(X, mask):
    A, B = X[:, mask], X[:, ~mask]
    n1, n2 = A.shape[1], B.shape[1]
    m1, m2 = A.mean(1), B.mean(1)
    sse = ((A - m1[:, None]) ** 2).sum(1) + ((B - m2[:, None]) ** 2).sum(1)
    se = np.sqrt((1 / n1 + 1 / n2) * sse / (n1 + n2 - 2))
    return m1 - m2, np.maximum(se, 1e-12)

diff_obs, se_obs = diff_and_se(X, obs_mask)


# ---------- 3. Choose the fudge factor s0 ----------
def choose_s0(diff, se, n_bins=None):
    n = len(se)
    n_bins = n_bins or int(np.clip(n // 20, 10, 100))
    cands = np.unique(np.concatenate([[0.0], np.percentile(se, np.arange(5, 101, 5))]))
    edges = np.quantile(se, np.linspace(0, 1, n_bins + 1))
    bins = np.clip(np.searchsorted(edges, se, side='right') - 1, 0, n_bins - 1)
    cvs = []
    for s0 in cands:
        d = diff / (se + s0)
        mads = []
        for b in range(n_bins):
            db = d[bins == b]
            if len(db) > 2:
                mads.append(1.4826 * np.median(np.abs(db - np.median(db))))
        mads = np.array(mads)
        cvs.append(mads.std() / mads.mean() if len(mads) > 1 and mads.mean() > 0 else np.inf)
    cvs = np.array(cvs)
    return cands[np.argmin(cvs)]

s0 = choose_s0(diff_obs, se_obs)
print(f"s0 = {s0:.3f}   (median SE = {np.median(se_obs):.3f})")

d_obs = diff_obs / (se_obs + s0)                # the SAM statistic


# ---------- 4. All unique relabellings of the samples ----------
perm_masks = []
for idx in combinations(range(n_samp), obs_mask.sum()):
    m = np.zeros(n_samp, bool)
    m[list(idx)] = True
    if not np.array_equal(m, obs_mask):
        perm_masks.append(m)
n_perm = len(perm_masks)
print(f"{n_perm} permutations (every possible relabelling except the observed one)")

d_perm = np.empty((n_perm, n_pairs))
for k, m in enumerate(perm_masks):
    dp, sp = diff_and_se(X, m)
    d_perm[k] = dp / (sp + s0)                  # same s0 as observed


# ---------- 5. Null proportion pi0 ----------
q25, q75 = np.percentile(d_perm, [25, 75])
pi0 = min(np.mean((d_obs > q25) & (d_obs < q75)) / 0.5, 1.0)
print(f"pi0 = {pi0:.2f}")


# ---------- 6. Per-pair q-values from the pooled permutation null ----------
a_obs = np.abs(d_obs)
a_null = np.sort(np.abs(d_perm).ravel())

# expected false positives at threshold |d_i| (+1 avoids reporting exactly 0)
fp = (len(a_null) - np.searchsorted(a_null, a_obs, side='left') + 1) / n_perm
# number of observed pairs at least this extreme
called = len(a_obs) - np.searchsorted(np.sort(a_obs), a_obs, side='left')

fdr = np.minimum(pi0 * fp / called, 1.0)
order = np.argsort(a_obs)                       # ascending |d|
q = np.empty_like(fdr)
q[order] = np.minimum.accumulate(fdr[order])    # enforce monotone q-values


# ---------- 7. Results ----------
res_sam = pd.DataFrame({
    'mean_g1': X[:, obs_mask].mean(1),
    'mean_g2': X[:, ~obs_mask].mean(1),
    'diff': diff_obs,
    'se': se_obs,
    'd_sam': d_obs,
    'q': q,
}, index=tab_c.index).iloc[np.argsort(-a_obs)]

print((res_sam['q'] < 0.05).sum(), "pairs with q < 0.05")
res_sam.head(20)


# ---------- 8. SAM plot: observed vs expected under the null ----------
d_sorted_idx = np.argsort(d_obs)
d_sorted = d_obs[d_sorted_idx]
d_exp = np.sort(d_perm, axis=1).mean(axis=0)    # expected order statistics
hit = q[d_sorted_idx] < 0.05

plt.figure(figsize=(5, 5))
plt.scatter(d_exp[~hit], d_sorted[~hit], s=4, c='grey')
plt.scatter(d_exp[hit], d_sorted[hit], s=8, c='red')
lim = [min(d_exp.min(), d_sorted.min()), max(d_exp.max(), d_sorted.max())]
plt.plot(lim, lim, 'k--', lw=0.8)
plt.xlabel('expected d (permutations)'); plt.ylabel('observed d')
plt.show()


# ---------- 9. Compare with the moderated t-test ----------


In [ ]:
import anndata as ad
import pandas as pd

# 1. Load the saved SpatialDM-processed files
names = ['A1', 'A2', 'A3', 'A4', 'A8', 'A9']
samples = {name: ad.read_h5ad(f'{name}_spatialdm.h5ad') for name in names}

# 2. Confirm the column name to pull — check this once before building the table
print(samples['A1'].uns['global_res'].columns.tolist())
print(samples['A1'].uns['global_res'].head())

In [ ]:
# 3. Build the (LR pair x sample) table SAM expects
stat_col = 'z'   # adjust based on what step 2 printed

tab = pd.DataFrame({
    name: ad.uns['global_res'][stat_col]
    for name, ad in samples.items()
})

print(tab.shape)
tab.head()

In [ ]:
# 4. Define your groups — adjust to your actual adult/fetal split
g1 = ['A1', 'A2']                 # adult
g2 = ['A3', 'A4', 'A8', 'A9']     # fetal

In [ ]:
# 1. Global shift: are adult z-scores higher across the board?
print(tab_c.median())                       # per-sample median z
print(res_sam['diff'].median())             # median difference across all pairs
print({n: ad.n_obs for n, ad in samples.items()})   # spot counts

# 2. Which pairs did SAM drop because of one-sided presence?
dropped = tab.loc[tab.index.difference(tab_c.index)]
adult_only = dropped[dropped[g1].notna().all(axis=1) & dropped[g2].isna().all(axis=1)]
fetal_only = dropped[dropped[g2].notna().all(axis=1) & dropped[g1].isna().all(axis=1)]
print(len(adult_only), len(fetal_only))

# 3. Pairs supported by both methods
both = set(res_sam[res_sam['q'] < 0.05].index) & set(res_mod[res_mod['fdr'] < 0.05].index)
print(len(both)); sorted(both)

In [ ]:
spatialdm_tab = pd.read_csv('spatialdm_results.csv', index_col=0)
commot_tab = pd.read_csv('commot_results.csv', index_col=0)
stlearn_tab = pd.read_csv('stlearn_results.csv', index_col=0)

method_tabs = {'spatialdm': spatialdm_tab, 'commot': commot_tab, 'stlearn': stlearn_tab}

In [ ]:
def zscore_tab(tab):
    return (tab - np.nanmean(tab.values)) / np.nanstd(tab.values)

spatialdm_tab_z = zscore_tab(spatialdm_tab)
commot_tab_z = zscore_tab(commot_tab)
stlearn_tab_z = zscore_tab(stlearn_tab)

method_tabs = {
    'spatialdm': spatialdm_tab_z,
    'commot': commot_tab_z,
    'stlearn': stlearn_tab_z,
}

In [ ]:
import numpy as np
import pandas as pd
from scipy import special, optimize
from itertools import combinations


# =========================================================================
# 1. BUILD THE MULTIVARIATE INPUT
# =========================================================================
# You should have one (LR-pair x sample) table per method, e.g.
#   spatialdm_tab, commot_tab, stlearn_tab  -- each a DataFrame,
#   rows = LR pair names, columns = sample names.
# This aligns them into one dict: pair_name -> DataFrame(samples x methods)

def build_multivariate_table(method_tabs: dict, cols):
    methods = list(method_tabs.keys())
    common_pairs = set(method_tabs[methods[0]].index)
    for m in methods[1:]:
        common_pairs &= set(method_tabs[m].index)
    common_pairs = sorted(common_pairs)

    multi = {}
    for pair in common_pairs:
        row = pd.DataFrame({m: method_tabs[m].loc[pair, cols] for m in methods})
        multi[pair] = row.dropna()        # complete cases only, per pair
    # drop pairs that lost too many samples to missingness
    multi = {k: v for k, v in multi.items() if len(v) == len(cols)}
    return multi, sorted(multi.keys())


# =========================================================================
# 2. PER-PAIR MEAN-DIFFERENCE VECTOR AND POOLED COVARIANCE
# =========================================================================

def pair_stats(df, g1, g2):
    A = df.loc[g1].to_numpy(float)     # (n1, p)
    B = df.loc[g2].to_numpy(float)     # (n2, p)
    n1, n2 = A.shape[0], B.shape[0]
    m1, m2 = A.mean(0), B.mean(0)
    diff = m1 - m2
    Ac, Bc = A - m1, B - m2
    scatter = Ac.T @ Ac + Bc.T @ Bc            # (p, p)
    d_resid = n1 + n2 - 2
    S = scatter / d_resid if d_resid > 0 else np.full(scatter.shape, np.nan)
    return n1, n2, diff, S, d_resid


def compute_all_pair_stats(multi, pairs, g1, g2):
    diffs, covs, dfs = {}, {}, {}
    for pair in pairs:
        _, _, diff, S, d_resid = pair_stats(multi[pair], g1, g2)
        diffs[pair], covs[pair], dfs[pair] = diff, S, d_resid
    return diffs, covs, dfs


# =========================================================================
# 3. ESTIMATE THE SHARED COVARIANCE PRIOR (S0, d0)
# =========================================================================
# Generalizes your moderated-t-test's fit_prior() to matrices, using the
# Bartlett decomposition of the Wishart log-determinant -- the matrix
# analogue of the log-chi-squared trick used for the scalar s0^2/d0.

def fit_matrix_prior(covs, dfs, p):
    pairs = [k for k in covs if dfs[k] > p]      # need d_i > p for det(S) to behave
    d_arr = np.array([dfs[k] for k in pairs], float)

    S_stack = np.stack([covs[k] for k in pairs])
    w = d_arr / d_arr.sum()
    S0 = np.tensordot(w, S_stack, axes=(0, 0))    # df-weighted average target matrix

    logdetS = np.array([np.log(max(np.linalg.det(covs[k]), 1e-300)) for k in pairs])

    def bartlett_mean(d):
        return sum(special.digamma((d - k + 1) / 2) - np.log(d / 2) for k in range(1, p + 1))

    def bartlett_var(d):
        return sum(special.polygamma(1, (d - k + 1) / 2) for k in range(1, p + 1))

    e = logdetS - np.array([bartlett_mean(d) for d in d_arr])
    evar = e.var(ddof=1) - np.mean([bartlett_var(d) for d in d_arr])

    if evar > 0:
        def f(d0):
            return sum(special.polygamma(1, (d0 - k + 1) / 2) for k in range(1, p + 1)) - evar
        try:
            d0 = optimize.brentq(f, p + 1e-3, 1e6)
        except ValueError:
            d0 = np.inf
    else:
        d0 = np.inf

    return S0, d0


# =========================================================================
# 4. SHRINKAGE COVARIANCE AND HOTELLING T^2
# =========================================================================

def shrink_cov(S_i, d_i, S0, d0, p, ridge=1e-6):
    S_tilde = (d0 * S0 + d_i * S_i) / (d0 + d_i) if np.isfinite(d0) else S0.copy()
    S_tilde += ridge * np.trace(S_tilde) / p * np.eye(p)   # numerical safety
    return S_tilde


def hotelling_t2(diff, S_tilde, n1, n2):
    S_inv = np.linalg.pinv(S_tilde)     # pseudo-inverse: safe near-singular
    return (n1 * n2 / (n1 + n2)) * diff @ S_inv @ diff


# =========================================================================
# 5. PERMUTATION NULL (same structure as your SAM code)
# =========================================================================

def all_relabelings(samples, n1):
    obs = set(samples[:n1])
    return [set(c) for c in combinations(samples, n1) if set(c) != obs]


def run_multivariate_sam(multi, pairs, g1, g2, p):
    samples = g1 + g2
    n1 = len(g1)

    # fit the shrinkage prior ONCE, from the real labels
    diffs0, covs0, dfs0 = compute_all_pair_stats(multi, pairs, g1, g2)
    S0, d0 = fit_matrix_prior(covs0, dfs0, p)
    print(f"prior df d0 = {d0:.2f}")

    # observed T^2 for every pair, using the real labels
    T2_obs, diff_obs = {}, {}
    for pair in pairs:
        n1o, n2o, diff, S_i, d_i = pair_stats(multi[pair], g1, g2)
        S_tilde = shrink_cov(S_i, d_i, S0, d0, p)
        T2_obs[pair] = hotelling_t2(diff, S_tilde, n1o, n2o)
        diff_obs[pair] = diff

    # every other relabeling, same fixed (S0, d0)
    masks = all_relabelings(samples, n1)
    print(f"{len(masks)} permutations")
    T2_perm = np.empty((len(masks), len(pairs)))
    for k, g1set in enumerate(masks):
        g1p = [s for s in samples if s in g1set]
        g2p = [s for s in samples if s not in g1set]
        for j, pair in enumerate(pairs):
            n1p, n2p, diffp, Sp, dp = pair_stats(multi[pair], g1p, g2p)
            Sp_tilde = shrink_cov(Sp, dp, S0, d0, p)
            T2_perm[k, j] = hotelling_t2(diffp, Sp_tilde, n1p, n2p)

    # pi0, then pooled FDR (T^2 already >= 0, no abs() needed)
    null_flat = T2_perm.ravel()
    obs_arr = np.array([T2_obs[pair] for pair in pairs])
    pi0 = min(np.mean(obs_arr < np.percentile(null_flat, 50)) / 0.5, 1.0)

    null_sorted = np.sort(null_flat)
    n_perm = T2_perm.shape[0]
    fp = (len(null_sorted) - np.searchsorted(null_sorted, obs_arr, side='left') + 1) / n_perm
    called = len(obs_arr) - np.searchsorted(np.sort(obs_arr), obs_arr, side='left')
    fdr = np.minimum(pi0 * fp / called, 1.0)

    order = np.argsort(obs_arr)
    q = np.empty_like(fdr)
    q[order] = np.minimum.accumulate(fdr[order])

    res = pd.DataFrame({'T2': obs_arr, 'q': q}, index=pairs)
    method_names = multi[pairs[0]].columns.tolist()
    for i, m in enumerate(method_names):
        res[f'diff_{m}'] = [diff_obs[pair][i] for pair in pairs]

    return res.sort_values('T2', ascending=False), S0, d0


# =========================================================================
# USAGE
# =========================================================================
# method_tabs = {
#     'spatialdm': spatialdm_tab,   # LR pairs (rows) x 6 samples (cols)
#     'commot':    commot_tab,
#     'stlearn':   stlearn_tab,
# }
g1 = ['A1', 'A2']                 # adult
g2 = ['A3', 'A4', 'A8', 'A9']     # fetal
#
multi, pairs = build_multivariate_table(method_tabs, g1 + g2)
p = len(method_tabs)
res_multi, S0, d0 = run_multivariate_sam(multi, pairs, g1, g2, p)
res_multi.head(20)


In [ ]:
print(len(pairs))
print(len(multi))

In [ ]:
sd_idx = set(spatialdm_tab_z.index)
cm_idx = set(commot_tab_z.index)
st_idx = set(stlearn_tab_z.index)

print('spatialdm ∩ commot:', len(sd_idx & cm_idx))
print('spatialdm ∩ stlearn:', len(sd_idx & st_idx))
print('commot ∩ stlearn:', len(cm_idx & st_idx))
print('all three:', len(sd_idx & cm_idx & st_idx))

In [ ]:
multi, pairs = build_multivariate_table(method_tabs, g1 + g2)
print(len(pairs))   # check this first before proceeding

In [ ]:
# diagnostic: for each of the 17 shared pairs, how many of the 6 samples are complete across all 3 methods?
for pair in common17:
    row = pd.DataFrame({m: method_tabs[m].loc[pair, g1+g2] for m in method_tabs})
    print(pair, row.dropna().shape[0])

In [ ]:
p = len(method_tabs)
res_multi, S0, d0 = run_multivariate_sam(multi, pairs, g1, g2, p)
print(f"d0 = {d0:.2f}")
res_multi.head(13)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------- 1. SAM plot (you already have this, reproduced for reference) ----------
def sam_plot(res_sam, d_perm):
    d_obs = res_sam['d_sam'].values
    idx = np.argsort(d_obs)
    d_sorted = d_obs[idx]
    d_exp = np.sort(d_perm, axis=1).mean(axis=0)
    hit = res_sam['q'].values[idx] < 0.05

    plt.figure(figsize=(5, 5))
    plt.scatter(d_exp[~hit], d_sorted[~hit], s=4, c='grey', label='not significant')
    plt.scatter(d_exp[hit], d_sorted[hit], s=10, c='red', label='q < 0.05')
    lim = [min(d_exp.min(), d_sorted.min()), max(d_exp.max(), d_sorted.max())]
    plt.plot(lim, lim, 'k--', lw=0.8)
    plt.xlabel('expected d (permutations)'); plt.ylabel('observed d')
    plt.legend(); plt.title('SAM plot'); plt.tight_layout(); plt.show()


# ---------- 2. Ranked bar chart, SAM: diff and direction in one view ----------
def ranked_bar(res, value_col, label_col=None, title='', top_n=20):
    sub = res.reindex(res[value_col].abs().sort_values(ascending=False).index).head(top_n)
    colors = ['#d62728' if v > 0 else '#1f77b4' for v in sub[value_col]]
    plt.figure(figsize=(6, max(4, 0.3 * len(sub))))
    plt.barh(sub.index[::-1], sub[value_col][::-1], color=colors[::-1])
    plt.axvline(0, color='black', lw=0.8)
    plt.xlabel(value_col); plt.title(title)
    plt.tight_layout(); plt.show()

ranked_bar(res_sam, 'diff', title='SAM: top LR pairs by effect size (red = adult-higher)')


# ---------- 3. Volcano-style plot: effect size vs significance, for either method ----------
def volcano(res, diff_col, sig_col, sig_is_q=True, title=''):
    x = res[diff_col]
    y = -np.log10(res[sig_col].clip(lower=1e-10))
    sig = res[sig_col] < 0.05
    plt.figure(figsize=(6, 5))
    plt.scatter(x[~sig], y[~sig], s=10, c='grey', alpha=0.6)
    plt.scatter(x[sig], y[sig], s=20, c='red')
    for pair in res[sig].index[:10]:          # label the top hits
        plt.annotate(pair, (x[pair], y[pair]), fontsize=7)
    plt.axhline(-np.log10(0.05), color='black', ls='--', lw=0.6)
    plt.xlabel('effect size (diff)'); plt.ylabel(f'-log10({sig_col})')
    plt.title(title); plt.tight_layout(); plt.show()

volcano(res_sam, 'diff', 'q', title='SAM volcano plot')


# ---------- 4. Multivariate T2: ranked bar using each method's diff ----------
# Since T2 has no single sign, show per-method direction as a small multiples / stacked view
def multivariate_direction_plot(res_multi, method_cols, top_n=13):
    sub = res_multi.sort_values('T2', ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(7, max(4, 0.4 * len(sub))))
    y = np.arange(len(sub))
    width = 0.8 / len(method_cols)
    for i, col in enumerate(method_cols):
        ax.barh(y + i * width, sub[col], height=width, label=col.replace('diff_', ''))
    ax.set_yticks(y + width * (len(method_cols) - 1) / 2)
    ax.set_yticklabels(sub.index)
    ax.axvline(0, color='black', lw=0.8)
    ax.invert_yaxis()
    ax.set_xlabel('z-scored difference (adult - fetal)')
    ax.legend()
    ax.set_title('Direction per method, ranked by multivariate T²')
    plt.tight_layout(); plt.show()

method_cols = [c for c in res_multi.columns if c.startswith('diff_')]
multivariate_direction_plot(res_multi, method_cols)


# ---------- 5. Agreement check: SAM ranking vs multivariate T2 ranking ----------
def rank_comparison(res_sam, res_multi):
    common = res_sam.index.intersection(res_multi.index)
    sam_rank = res_sam.loc[common, 'd_sam'].abs().rank(ascending=False)
    multi_rank = res_multi.loc[common, 'T2'].rank(ascending=False)

    plt.figure(figsize=(5, 5))
    plt.scatter(sam_rank, multi_rank)
    for pair in common:
        plt.annotate(pair, (sam_rank[pair], multi_rank[pair]), fontsize=7)
    lim = [1, max(sam_rank.max(), multi_rank.max())]
    plt.plot(lim, lim, 'k--', lw=0.6)
    plt.xlabel('SAM rank (|d_sam|)'); plt.ylabel('Multivariate T² rank')
    plt.title('Agreement between univariate SAM and multivariate T²')
    plt.tight_layout(); plt.show()

rank_comparison(res_sam, res_multi)

In [ ]:
for name, tab in [('spatialdm', spatialdm_tab_z), ('commot', commot_tab_z), ('stlearn', stlearn_tab_z)]:
    dupes = tab.index[tab.index.duplicated()]
    print(name, 'duplicate pair names:', len(dupes))
    if len(dupes) > 0:
        print(dupes.unique()[:10])

In [ ]:
print(p, type(p))

In [ ]:
p = len(method_tabs)
print(p, type(p))   # confirm it's now 3 <class 'int'>

In [ ]:
# Recompute T2_obs and T2_perm directly, reusing the already-fitted S0, d0

samples_list = g1 + g2
n1 = len(g1)

T2_obs = {}
for pair in pairs:
    n1o, n2o, diff, S_i, d_i = pair_stats(multi[pair], g1, g2)
    S_tilde = shrink_cov(S_i, d_i, S0, d0, p)
    T2_obs[pair] = hotelling_t2(diff, S_tilde, n1o, n2o)

masks = all_relabelings(samples_list, n1)
T2_perm = np.empty((len(masks), len(pairs)))
for k, mset in enumerate(masks):
    g1p = [s for s in samples_list if s in mset]
    g2p = [s for s in samples_list if s not in mset]
    for j, pair in enumerate(pairs):
        n1p, n2p, diffp, Sp, dp = pair_stats(multi[pair], g1p, g2p)
        Sp_tilde = shrink_cov(Sp, dp, S0, d0, p)
        T2_perm[k, j] = hotelling_t2(diffp, Sp_tilde, n1p, n2p)

obs_arr = np.array([T2_obs[pair] for pair in pairs])

print("max permuted T2 (all pairs, all permutations):", T2_perm.max())
print("observed T2 range:", obs_arr.min(), "-", obs_arr.max())

null_flat = T2_perm.ravel()
q25, q75 = np.percentile(null_flat, [25, 75])
pi0 = min(np.mean((obs_arr > q25) & (obs_arr < q75)) / 0.5, 1.0)
print("pi0 estimate:", pi0)

Running COMMOT

In [ ]:
import commot as ct


In [ ]:
df_ligrec = ct.pp.ligand_receptor_database(database='CellChat', species='human')
print(df_ligrec.shape)
df_ligrec.head()

In [ ]:
df_ligrec_filt = ct.pp.filter_lr_database(df_ligrec, A1, min_cell_pct=0.05)
print(df_ligrec_filt.shape)

In [ ]:
samples = {'A1': A1, 'A2': A2, 'A3': A3, 'A4': A4, 'A8': A8, 'A9': A9}

for name, ad in samples.items():
    ct.tl.spatial_communication(
        ad, database_name='cellchat', df_ligrec=df_ligrec_filt,
        dis_thr=500, heteromeric=True, pathway_sum=True,
    )
    print(f'{name} done')

In [ ]:
def commot_pair_totals(ad, df_ligrec):
    totals = {}
    for _, row in df_ligrec.iterrows():
        ligand, receptor = row[0], row[1]
        pair_name = f'{ligand}_{receptor}'
        key = f'commot-cellchat-{ligand}-{receptor}'
        if key in ad.obsp:
            totals[pair_name] = ad.obsp[key].sum()
    return pd.Series(totals)

In [ ]:
print(df_ligrec_filt.index[:5])
print(df_ligrec_filt.columns.tolist())
print(df_ligrec_filt.head())

In [ ]:
print([k for k in A1.obsp.keys() if k.startswith('commot')][:10])

In [ ]:
commot_tab = pd.DataFrame({
    name: commot_pair_totals(ad, df_ligrec_filt)      # <-- df_ligrec_filt, not .index
    for name, ad in samples.items()
})
commot_tab.head()

In [ ]:
commot_tab.to_csv('commot_results.csv')

In [ ]:
for name, ad in samples.items():
    try:
        ad.write_h5ad(f'{name}_commot.h5ad')
        print(f'{name} saved')
    except Exception as e:
        print(f'{name} FAILED: {type(e).__name__}: {e}')
        # if it fails, apply fix_uns(ad) the same way as before and retry

In [ ]:
import stlearn as st
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

np.random.seed(seed)
samples_raw = {
    'A1': A1, 'A2': A2, 'A3': A3, 'A4': A4, 'A8': A8, 'A9': A9,
}  # these should be freshly loaded/raw AnnData, not the SpatialDM-processed ones

lrs = st.tl.cci.load_lrs(['connectomeDB2020_lit'], species='human')

seed = 42
stlearn_results = {}

for name, adata in samples_raw.items():
    print(f'--- {name} ---')

    # preprocessing (same as your original stlearn run)
    st.pp.filter_genes(adata, min_cells=3)
    st.pp.normalize_total(adata)   # no log1p, per stlearn's own convention

    # non-Visium spatial setup (from earlier in this conversation)
    coords = adata.obsm['spatial']
    adata.obs['imagerow'] = coords[:, 1] if not hasattr(coords, 'iloc') else coords.iloc[:, 1].values
    adata.obs['imagecol'] = coords[:, 0] if not hasattr(coords, 'iloc') else coords.iloc[:, 0].values

    # estimate a sensible neighbor distance from actual spot spacing
    nbrs = NearestNeighbors(n_neighbors=6).fit(coords)
    dists, _ = nbrs.kneighbors(coords)
    mean_nn_dist = dists[:, 1:].mean()
    distance = mean_nn_dist * 1.5

    # run CCI
    st.tl.cci.run(adata, lrs,
                  min_spots=20,
                  distance=distance,
                  n_pairs=100,
                  n_cpus=None)

    stlearn_results[name] = adata
    print(f'{name} done, distance used = {distance:.2f}')

In [ ]:
print(A1.uns['lr_summary'].columns.tolist())
print(A1.uns['lr_summary'].head())

In [ ]:
import pandas as pd

stat_col = 'n_spots_sig'   # or 'n_spots_sig_pval' — see note below on which to pick

stlearn_tab = pd.DataFrame({
    name: ad.uns['lr_summary'][stat_col] / ad.uns['lr_summary']['n_spots']
    for name, ad in stlearn_results.items()
})
stlearn_tab.head()


In [ ]:
print((A1.uns['lr_summary']['n_spots_sig_pval'] >= A1.uns['lr_summary']['n_spots_sig']).mean())

In [ ]:
stlearn_tab.to_csv('stlearn_results.csv')